In [ ]:
import gc
import os
import sys
sys.path.append("/workspace/mta_vision_transformers/")
from collections import OrderedDict
from typing import Any, Callable, Dict, List, Set, Tuple

import numpy as np
import einops
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import torch.utils.data
from matplotlib import pyplot as plt
from tensordict import TensorDict
from torch.utils._pytree import tree_flatten

from core.monitor import Monitor
from infrastructure import utils
from infrastructure.settings import DEVICE, OUTPUT_DEVICE, DTYPE
from dataset.construct import ImageDataset
from dataset.library import DATASETS


dataset_name, n_classes = DATASETS["Common"][1]
OUTPUT_DIR = "experiments/attention"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
# Ocean: 901085904
# Rose: 100390212
torch.set_printoptions(linewidth=400, sci_mode=False)

In [ ]:
from modeling.vit_attention import OpenCLIPAttentionViT
from visualize.base import construct_per_layer_output_dict


# SECTION: Set up model
torch.set_default_device(DEVICE)
model = OpenCLIPAttentionViT("default", 0, None).to(DEVICE)


# SECTION: Set up monitor
def residual_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return input_ + tree_flatten(output_)[0][0]
    
def input_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return tree_flatten(input_)[0][0]

def attention_matrix_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return torch.mean(einops.rearrange(
        tree_flatten(output_)[0][0],
        "b h n1 n2 -> b n1 n2 h"
    ), dim=-1).to(OUTPUT_DEVICE)


monitor_config = OrderedDict({
    "model.visual.transformer.resblocks": OrderedDict({
        "": [
            # ("layer_input", input_hook_fn),
            ("layer_output", Monitor.default_hook_fn),
        ],
        # "ln_1": "layer_norm1_output",  # "norm1"
        "return_attn_matrix": [
            ("attention_matrix", attention_matrix_hook_fn)
        ],
        "attn": [
            # ("attention_input", input_hook_fn),
            # ("query", query_hook_fn),
            # ("key", key_hook_fn),
            # ("attention_proj", attention_proj_hook_fn),
            # ("attention_output", Monitor.default_hook_fn),
            # ("attention_matrix", attention_matrix_hook_fn),
        ],
    })
})

monitor = Monitor(model, monitor_config)


# SECTION: Set up dataset
batch_size = 50
dataset = ImageDataset(dataset_name, split="train", return_original_image=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator(DEVICE))
original_images, images = next(iter(dataloader))
# original_images, images = torch.flip(original_images, dims=[0]), torch.flip(images, dims=[0])


# SECTION: Run baseline model
per_metric_output_dict = monitor.reset()
with torch.no_grad():
    output = model.forward(images)

per_layer_output_dict = construct_per_layer_output_dict(per_metric_output_dict)

In [ ]:
# SECTION: Visualize original images
%matplotlib inline
from modeling.image_features import ImageFeatures
from core.attention_sink import mask_attention_sink
from visualize.base import (
    visualize_images_with_mta,
    get_rgb_colors,
)

torch.set_default_device(OUTPUT_DEVICE)

# SECTION: Massive token heuristic
detection_layer = 13

artifact_mask: torch.Tensor = torch.load(f"{OUTPUT_DIR}/ranked_AS_mask.pt", map_location=OUTPUT_DEVICE) > 0
mta_masks: Dict[int, torch.Tensor] = {
    "MA": mask_attention_sink(
        per_layer_output_dict[detection_layer]["attention_matrix"],
        max_num_tokens=None, scale=0.4, verbose=True
    ),
    "Artifact": artifact_mask
}
features = ImageFeatures(per_layer_output_dict, mta_masks, "default", DEVICE)

for k, v in mta_masks.items():
    print(f"{k}: {v.sum().item()}/{v.numel()}")


# SECTION: Visualize images
for mask in mta_masks.values():
    visualize_images_with_mta(original_images.to(OUTPUT_DEVICE), mask.to(OUTPUT_DEVICE))

try:
    rgb_assignment
except NameError:
    color_layer_idx = 10    # min(mta_masks.keys())
    rgb_assignment = get_rgb_colors(features, color_layer_idx, "layer_output", False)

# highlight = torch.LongTensor((
#     (1, 5, 4),
#     (4, 15, 8),
# ))
# highlight = torch.load(f"{OUTPUT_DIR}/artifact_indices.pt")

In [ ]:
%matplotlib inline
from core.attention_sink import mask_attention_sink
from modeling.image_features import ImageFeatures
from visualize.base import VISUALIZED_INDICES, visualize_features_per_image, construct_per_layer_output_dict
from visualize.attention import visualize_attention_matrix_per_image, visualize_incoming_attention_per_image


save_experiment = False
mode = "mask"
mask_layer = 9
stop_layer = detection_layer + 1

attention_matrix: torch.Tensor = per_layer_output_dict[detection_layer]["attention_matrix"].to(DEVICE)
plotting_kwargs: Dict[str, Any] = {
    "transform_func": None,
    "per_head": False,
    "rescale_func": lambda t: torch.log2(t + 1),
    "global_cmap": False,
    "cmap_scale": "linear",
    "subsample": 1.0,
    "spacing": 0.1,
    "cmap": "viridis",
}
visualize_attention_matrix_per_image(detection_layer, attention_matrix, mta_masks, **plotting_kwargs)
# visualize_incoming_attention_per_image(detection_layer, attention_matrix, cmap="gray")

highlight_mask = torch.full((batch_size, ImageFeatures.N + 1), False)
highlight_mask[:, [67, 126, 222]] = True
# for c in [(3, 67), (3, 126), (3, 222)]:
#     highlight_mask[c] = True

fname = f"{OUTPUT_DIR}/ranked_AS_mask.pt"
if save_experiment and os.path.exists(fname):
    ranked_AS_mask: torch.Tensor = torch.load(fname, map_location=OUTPUT_DEVICE)

else:
    cache: List[torch.Tensor] = [
        per_layer_output_dict[layer_idx]["layer_output"]
        for layer_idx in range(mask_layer)
    ]
    
    AS_mask: torch.Tensor = torch.full((batch_size, ImageFeatures.N + 1), False)
    ranked_AS_mask: torch.Tensor = torch.zeros((batch_size, ImageFeatures.N + 1), dtype=torch.int)
    with torch.no_grad():
        it, max_it, start_vis_it = 1, float("inf"), float("inf")
        convergence = torch.full((batch_size,), False)
        while not torch.all(convergence):
            if it > max_it:
                break
            
            s = "".join("\u25A0" if c else "\u25A1" for c in convergence)    
            print("=" * 160)
            print(f"Iteration {it}: {s}")
            print("=" * 160)
            
            torch.set_default_device(DEVICE)

            new_model = OpenCLIPAttentionViT(mask_layer, AS_mask, mode, cache=cache, stop_layer=stop_layer).to(DEVICE)
            new_monitor = Monitor(new_model, monitor_config)

            new_per_metric_output_dict = new_monitor.reset()
            new_output = new_model.forward(images)

            new_per_layer_output_dict: List[TensorDict] = [None] * mask_layer + construct_per_layer_output_dict({
                k: v[mask_layer:]
                for k, v in new_per_metric_output_dict.items()
            })

            torch.set_default_device(OUTPUT_DEVICE)

            # SECTION: Massive token heuristic
            new_attention_matrix = new_per_layer_output_dict[detection_layer]["attention_matrix"].to(DEVICE)
            
            # SECTION: Visualize the attention matrix before and after
            new_MA_mask = mask_attention_sink(new_attention_matrix.to(OUTPUT_DEVICE), masked_tokens=AS_mask, verbose=True)
            
            if it >= start_vis_it:
                print(f"\tNumber of masked MAs: {torch.sum(AS_mask, dim=1)[VISUALIZED_INDICES].tolist()}")
                print(f"\tNumber of new MAs: {torch.sum(new_MA_mask, dim=1)[VISUALIZED_INDICES].tolist()}")
                
                visualize_attention_matrix_per_image(
                    detection_layer, new_attention_matrix, {
                        "": AS_mask,
                        "Artifact": mta_masks["Artifact"],
                        # "Highlight": highlight_mask,
                    }, **plotting_kwargs,
                )
                visualize_attention_matrix_per_image(
                    detection_layer, new_attention_matrix, {
                        "": AS_mask,
                        "AS": new_MA_mask,
                        "Artifact": mta_masks["Artifact"],
                        # "Highlight": highlight_mask,
                    }, **plotting_kwargs,
                )
                visualize_incoming_attention_per_image(detection_layer, new_attention_matrix, cmap="gray", global_cmap=False)
                visualize_features_per_image(features, 19, "layer_output", mta_mask=AS_mask, highlight=new_MA_mask)
            
            # SECTION: Update the cumulative MA mask with the new attention sinks
            AS_mask = AS_mask + new_MA_mask
            ranked_AS_mask = ranked_AS_mask + it * new_MA_mask
            it += 1
            
            # SECTION: Check convergence
            convergence = torch.sum(new_MA_mask, dim=1) == 0
            
            # SECTION: Cleanup
            del new_per_metric_output_dict, new_per_layer_output_dict, new_attention_matrix, new_MA_mask
            torch.cuda.empty_cache()
            gc.collect()

    # SECTION: Save results
    if save_experiment:
        torch.save(ranked_AS_mask, fname)
AS_mask: torch.Tensor = (ranked_AS_mask > 0)

In [ ]:
# SECTION: Compare attention sink type artifact tokens vs diagonal type artifact tokens
from visualize.base import visualize_features_per_image
from visualize.projections import visualize_feature_values_by_pca


diagonal_mask = artifact_mask * ~AS_mask
# torch.save(diagonal_mask, f"{OUTPUT_DIR}/diagonal_mask.pt")
print(torch.sum(AS_mask).item(), torch.sum(diagonal_mask).item())

m = torch.full((batch_size, ImageFeatures.N + 1), False)
m[-4, :] = True
new_ranked_AS_mask = ranked_AS_mask * m

visualize_features_per_image(features, 20, "layer_output", mta_mask=AS_mask, highlight=diagonal_mask)
visualize_feature_values_by_pca(
    features, 10, "layer_output", {"linear"}, new_ranked_AS_mask, rgb_assignment,
    ndim=2,
    with_cls=True,
    highlight=diagonal_mask,
    alpha=1.0,
)